In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

options = ['A', 'B', 'C', 'D', 'E']

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

In [6]:
def map_at_3(df, predict_fn):
    scores = []
    for _, row in df.iterrows():
        prediction = predict_fn(row)
        predicted_labels = prediction.split()
        correct = row['answer']
        score = 0.0
        if correct in predicted_labels:
            rank = predicted_labels.index(correct) + 1
            score = 1.0 / rank
        scores.append(score)
    return np.mean(scores)

In [3]:
# BAAI/bge-large-en-v1.5 performed best among sentence transformers
model = SentenceTransformer('BAAI/bge-large-en-v1.5')

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 399.83it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Models tried:
# - all-MiniLM-L6-v2: Kaggle MAP@3 = 0.38653
# - multi-qa-mpnet-base-dot-v1: performed worse than MiniLM
# - BAAI/bge-large-en-v1.5: Kaggle MAP@3 = 0.42310 (best)

In [4]:
def predict_top3_transformer(row):
    prompt_embedding = model.encode(row['prompt'])
    option_embeddings = model.encode([row[opt] for opt in options])
    scores = cosine_similarity([prompt_embedding], option_embeddings).flatten()
    top3_indices = scores.argsort()[::-1][:3]
    return ' '.join([options[i] for i in top3_indices])

In [7]:
sample = train.sample(200, random_state=42)
score = map_at_3(sample, predict_top3_transformer)
print(f"BAAI/bge-large Local MAP@3: {score:.4f}")
# Kaggle MAP@3: 0.42310 (V14)

BAAI/bge-large Local MAP@3: 0.4308


In [9]:
test['Prediction'] = test.apply(predict_top3_transformer, axis=1)
submission = test[['id', 'Prediction']].copy()
submission.columns = ['ID', 'Prediction']
submission.to_csv('sentence_transformer_submission.csv', index=False)
print(submission.head())

   ID Prediction
0   1      B D A
1   2      E A C
2   3      A D C
3   4      E C A
4   5      C D E
